# Rosetta — Recherche d'hyperparamètres Optuna (étape 8)

**Rôle de ce notebook** : mettre en place l'**ossature complète et exécutable** de la
recherche d'hyperparamètres Optuna décrite à l'étape 8 de `docs/plan-seq2seq.md`, avec
une couche d'**adaptateurs** vers les étapes 0→7 du pipeline (chargement/nettoyage/split,
vocabulaire, numérisation, modèle, boucle d'entraînement).

**Périmètre tranché** :
- la mécanique Optuna (espace de recherche, pruning, storage SQLite reprenable, logs,
  export des résultats, figures) est écrite **pour de vrai** ;
- le sous-échantillonnage 20 % stratifié par longueur (groupé par cible EN, cohérent
  avec l'anti-fuite de l'étape 3) est écrit **pour de vrai** ;
- les étapes 0→7 elles-mêmes (`src/data`, `src/tokenization`, `src/models`,
  `src/training`) ne sont **pas encore codées** (`src/` ne contient que des
  `__init__.py` vides) : ce notebook bascule automatiquement sur des **stubs
  fonctionnels** (corpus jouet, mini seq2seq torch) tant qu'elles n'existent pas.

**Ce que ce notebook NE fait PAS** :
- il ne remplace pas les étapes 0→7 : dès que `src/` expose les 5 symboles attendus
  (`loadSplits`, `buildVocabs`, `makeDataloaders`, `buildModel`, `trainAndValidate`),
  ce notebook les utilise automatiquement, sans aucune modification de code ;
- il ne mesure aucune qualité de traduction réelle tant que `src/` n'est pas branché :
  les résultats obtenus sur le corpus jouet ne valident que la **mécanique** Optuna ;
- il n'entraîne pas les modèles finaux (étape 9) ni ne calcule de métriques de
  traduction (étape 10) ;
- il ne touche pas à `requirements.txt` (réécrit uniquement par le notebook AED).

In [ ]:
# Paramètres
projectName = "Rosetta"  # nom du projet, utilisé dans les noms de study Optuna
randomSeed = 42  # graine globale (numpy, torch, sampler Optuna) pour reproductibilité

# Langues
srcLang = "fr"  # langue source (colonne du DataFrame)
tgtLang = "en"  # langue cible (colonne du DataFrame)

# Tokenisation (étape 4) -- config fixée par run, HORS de l'espace de recherche Optuna
# (cf. étapes 8 et 8bis : le tokeniseur change les données, pas seulement le modèle).
# Valeurs possibles : "full", "words95", "bpe4k", "bpe8k", "unigram4k", "unigram8k".
tokenizationConfig = "words95"

# Architectures comparées (option A du plan : une étude Optuna séparée par architecture)
architectures = ["rnn", "gru"]  # valeurs passées telles quelles à EncoderStub/DecoderStub

# Attention additive de Bahdanau (étape 9) : variante FIXÉE PAR RUN, PAS un hyperparamètre
# Optuna (elle change l'architecture, pas seulement son réglage -- cf. étape 8bis pour la
# même logique appliquée au tokeniseur). Mettre à True lance une étude "avec attention".
useAttention = False

# Forcer les stubs même si src/ expose déjà des symboles (utile tant que le pipeline
# est partiellement implémenté : évite de charger le corpus réel avec des budgets
# calibrés pour le corpus jouet).
forceStubs = False

# Longueur & sous-échantillonnage stratifié (étape 8 du plan)
longLengthThreshold = 18  # seuil en mots de la cible EN : strate courte/longue (cf. étape 3)
subsampleFraction = 0.20  # fraction de groupes conservée pour la recherche Optuna

# Budget d'entraînement par essai -- RECALIBRÉ pour le pipeline réel (mesure de coût
# ci-dessous, sous-échantillon 20 % réel, tokenizationConfig="words95", hiddenDim=256,
# embDim=128, val COMPLÈTE à chaque epoch -- voir rapport du lot normalisation+étapes 6-7) :
#   - 1 epoch mesurée (train + val complète) : RNN ~73s | GRU ~97s
maxEpochs = 8  # nb max d'epochs par essai (le pruning peut arrêter avant)
batchSize = 64  # taille de batch des DataLoader
patience = 2  # epochs sans amélioration de la val loss avant arrêt anticipé (hors pruning)
gradClip = 1.0  # norme max pour clip_grad_norm_ (les RNN ont besoin de ce clipping, étape 7)
teacherForcingRatio = 0.5  # probabilité de nourrir le décodeur avec le vrai token précédent

# Budget Optuna -- RECALIBRÉ pour le pipeline réel (le nTrials=10 précédent visait un
# smoke-test < 3 min sur corpus jouet). Avec maxEpochs=8 ci-dessus et nTrials=15 :
#   - 1 essai, PIRE CAS (sans pruning ni early stopping) : RNN ~= 8*73s ~= 9.7min
#     | GRU ~= 8*97s ~= 12.9min
#   - 1 étude (15 essais), PIRE CAS : RNN ~= 15*9.7min ~= 2.4h | GRU ~= 15*12.9min ~= 3.2h
#   - les 2 études (option A, une par architecture) : PIRE CAS ~= 5.6h au total, pour
#     tokenizationConfig="words95" SEUL. Le pruning (MedianPruner) et patience=2 réduisent
#     ce pire cas en pratique, mais l'ampleur n'a PAS été mesurée ici (aucune étude Optuna
#     complète lancée, conformément à la consigne).
#   - Le vocabulaire pèse lourd sur ce budget : la config "full" (35282 FR / 23365 EN)
#     coûte ~587s/epoch en GRU, ~6x le coût de "words95" (softmax de sortie dominé par la
#     taille du vocabulaire EN) -- extrapolé aux 6 configs de tokenisation (bpe8k/unigram8k
#     ~2.3x, bpe4k/unigram4k ~1.1x, full ~6.6x), les "12 études" évoquées dans l'État vivant
#     du projet (6 configs x 2 architectures) ne sont PAS réalistes en Optuna complet sur ce
#     poste CPU (pire cas très largement > 24h, potentiellement plusieurs jours).
#     RECOMMANDATION : garder la recherche Optuna complète sur "words95" seul (déjà le choix
#     de tokenizationConfig ci-dessus) ; traiter la comparaison des 6 tokenisations
#     (étape 8bis, optionnelle) comme des runs isolés à hyperparamètres fixés (1 essai, pas
#     une étude Optuna), pas comme 6 études supplémentaires.
nTrials = 15  # nombre d'essais Optuna par étude

# Espace de recherche (étape 8, restreint aux hyperparamètres qui ne changent pas les données)
lrRange = (1e-4, 1e-2)  # borne du learning rate, échantillonné en log-uniforme
hiddenDimChoices = [128, 256, 512]  # dimension cachée de la cellule récurrente, catégoriel
embDimChoices = [128, 256]  # dimension des embeddings, catégoriel
dropoutRange = (0.0, 0.5)  # borne du dropout
dropoutStep = 0.05  # pas de discrétisation du dropout (suggest_float(..., step=...))

# Pruning (MedianPruner)
nStartupTrials = 3  # essais initiaux jamais élagués, le temps d'avoir une médiane fiable
nWarmupSteps = 1  # epochs initiales non évaluées pour le pruning, au sein de chaque essai

# Chemins (relatifs à notebooks/, comme dans l'AED)
reportsDir = "../reports/optuna"  # CSV, JSON et figures produits par ce notebook
storageDir = "../checkpoints"  # storage SQLite Optuna (reprenable d'une exécution à l'autre)
studyStorageName = "optuna_rosetta.db"  # nom du fichier SQLite

# Corpus jouet (stub, désactivé dès que src/ expose les 5 symboles attendus -- c'est
# maintenant le cas : ces 2 valeurs ne sont plus utilisées qu'en repli (forceStubs=True),
# volontairement laissées "jouet" pour ce cas de smoke-test rapide, PAS recalibrées vers le
# corpus réel : maxLen réel est de toute façon dérivé automatiquement du p99, cf. étape 5)
toyCorpusSize = 3000  # nombre de paires FR-EN synthétiques générées (base, avant duplication)
maxLen = None  # None = auto (p99 des longueurs en tokens + 2 bornes, étape 5) ; le stub retombe sur une valeur jouet (16) si None

In [ ]:
# Imports et configuration globale
# Rendre `src/` importable quel que soit le dossier depuis lequel le kernel est
# lance (Jupyter demarre generalement dans notebooks/). Sans cela,
# resolvePipeline() ne detecterait jamais les modules du pipeline.
import sys
from pathlib import Path

projectRoot = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir()),
    Path.cwd(),
)
if str(projectRoot) not in sys.path:
    sys.path.insert(0, str(projectRoot))
import json
import math
import importlib
import warnings
from pathlib import Path
from typing import Callable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import optuna
import optuna.visualization.matplotlib as optunaViz
from optuna.importance import PedAnovaImportanceEvaluator

from IPython.display import display

warnings.filterwarnings("ignore")

np.random.seed(randomSeed)
torch.manual_seed(randomSeed)

device = torch.device("cpu")  # pas de GPU NVIDIA sur ce poste (cf. CLAUDE.md)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

print(f"Projet: {projectName}")
print(
    f"torch: {torch.__version__} | optuna: {optuna.__version__} | "
    f"pandas: {pd.__version__} | numpy: {np.__version__}"
)
print(f"Device: {device}")

## 1. Adaptateurs vers le pipeline (étapes 0→7)

`resolvePipeline()` tente d'importer, depuis `src/`, les symboles attendus pour
chaque étape -- recherchés en **snake_case** (convention `src/` du projet, ex.
`load_splits`), exposés ensuite comme variables locales **camelCase** dans ce
notebook (ex. `loadSplits`). La bascule est **tout ou rien** : si les 5 symboles
ne sont pas TOUS résolus depuis `src/`, le notebook utilise les stubs pour LES
CINQ (sinon un `loadSplits` réel tournerait avec des stubs calibrés pour le
corpus jouet -- `maxLen`, `toyCorpusSize`... -- incohérent et inutilisable). Le
tableau affiché récapitule, pour chaque symbole, le module cible, l'étape du plan
concernée et l'état effectif à trois valeurs : `src/ (utilisé)`,
`src/ (disponible, non utilisé)` ou `stub`.

In [ ]:
# Résolution du pipeline : src/ si disponible, sinon bascule sur les stubs (tout ou rien)
def resolvePipeline() -> dict:
    """
    Tente d'importer, depuis `src/`, les symboles réels des étapes 0→7 -- recherchés
    en **snake_case** (convention `src/` du projet : `load_splits`, `build_vocabs`,
    `make_dataloaders`, `build_model`, `train_and_validate`), pas en camelCase.
    Bascule **TOUT OU RIEN** : si les 5 symboles ne sont pas TOUS résolus depuis
    `src/`, le notebook utilise les stubs pour LES CINQ. Sans cette règle, un
    `loadSplits` réel (corpus 264k paires) tournerait avec des stubs calibrés pour
    le corpus jouet (`maxLen`, `toyCorpusSize`...) -- incohérent et très lent.

    Affiche un tableau d'état à trois valeurs par symbole :
      - "src/ (utilisé)"                 : résolu, ET les 5 le sont (bascule active)
      - "src/ (disponible, non utilisé)" : résolu, mais au moins un autre symbole manque
      - "stub" / "stub (forcé)"          : non résolu depuis src/, ou `forceStubs=True`
    """
    symbols = [
        ("loadSplits", "load_splits", "src.data.splits", "0-3 (chargement, nettoyage, split)"),
        ("buildVocabs", "build_vocabs", "src.tokenization.vocab_builder", "4 (vocabulaire)"),
        ("makeDataloaders", "make_dataloaders", "src.tokenization.numerize", "5 (numérisation)"),
        ("buildModel", "build_model", "src.models.seq2seq", "6 (encodeur-décodeur)"),
        ("trainAndValidate", "train_and_validate", "src.training.loop", "7 (boucle d'entraînement)"),
    ]

    resolved = {}
    availability = {}

    if not forceStubs:
        for camelName, snakeName, moduleName, _step in symbols:
            try:
                module = importlib.import_module(moduleName)
                resolved[camelName] = getattr(module, snakeName)
                availability[camelName] = True
            except (ImportError, AttributeError):
                availability[camelName] = False

    ready = (not forceStubs) and len(resolved) == len(symbols)

    rows = []
    for camelName, snakeName, moduleName, step in symbols:
        if forceStubs:
            etat = "stub (forcé)"
        elif availability[camelName] and ready:
            etat = "src/ (utilisé)"
        elif availability[camelName]:
            etat = "src/ (disponible, non utilisé)"
        else:
            etat = "stub"
        rows.append({"symbole": snakeName, "module": moduleName, "étape": step, "état": etat})

    display(pd.DataFrame(rows))
    # Tout ou rien : si le pipeline n'est pas prêt, aucun symbole résolu n'est exposé --
    # la cellule suivante (`pipelineInfo["resolved"].get(name, stub)`) retombe alors
    # systématiquement sur le stub, pour LES CINQ symboles, même si certains étaient
    # individuellement résolus depuis src/.
    return {"resolved": resolved if ready else {}, "ready": ready, "rows": rows}


pipelineInfo = resolvePipeline()
pipelineReady = pipelineInfo["ready"]

nPrets = sum(1 for row in pipelineInfo["rows"] if row["état"].startswith("src/"))
nTotal = len(pipelineInfo["rows"])

if not pipelineReady:
    print("=" * 74)
    print("ATTENTION : src/ n'est pas encore branché (étapes 0→7 non implémentées).")
    print("Les résultats ci-dessous viennent d'un CORPUS JOUET synthétique.")
    print("Ils valident la mécanique Optuna (espace, pruning, storage, reprise, logs,")
    print("figures) -- PAS la qualité de traduction d'un vrai pipeline FR->EN.")
    print(f"{nPrets} symbole(s) sur {nTotal} prêt(s) depuis src/ -- résolution TOUT OU RIEN :")
    print("les stubs restent utilisés pour LES CINQ tant qu'il n'y en a pas 5/5.")
    print("La bascule vers src/ se fera automatiquement dès que les 5 le seront.")
    print("=" * 74)
else:
    print("src/ est branché (5/5 symboles) : ce notebook utilise le pipeline réel.")

# Provenance des artefacts : ne jamais mélanger résultats jouet et résultats réels.
runReportsDir = reportsDir if pipelineReady else f"{reportsDir}/stub"
studyNameSuffix = "" if pipelineReady else "-stub"
print(f"Dossier de sortie effectif : {runReportsDir} (suffixe d'étude : '{studyNameSuffix}')")

In [ ]:
# Stub données (étapes 0→3) : corpus jouet FR→EN avec queue longue et cibles dupliquées
def generateToyCorpus(size: int, seed: int) -> pd.DataFrame:
    """
    Génère un corpus FR→EN synthétique : lexique bijectif bruité, longueurs variables
    avec une vraie queue longue (> longLengthThreshold mots), et cibles EN dupliquées
    (alignements 1→N) via des mots FR synonymes. Remplace fonctionnellement les
    étapes 0→3 tant que `src/data/splits.py` n'existe pas.
    """
    rng = np.random.default_rng(seed)

    vocabWords = 50  # taille du lexique bijectif de base
    frBase = [f"fr{i:02d}" for i in range(vocabWords)]
    enBase = [f"en{i:02d}" for i in range(vocabWords)]
    lexicon = dict(zip(frBase, enBase))  # bijection FR -> EN

    nSynonymPairs = 8  # nb de mots EN atteignables par 2 mots FR différents (1->N légitime)
    synonyms = {f"frsyn{i:02d}": f"en{i:02d}" for i in range(nSynonymPairs)}
    lexicon.update(synonyms)
    frVocabForSentences = frBase + list(synonyms.keys())

    noiseProb = 0.05  # probabilité qu'un mot FR soit traduit "au hasard" (bruit du lexique)

    def translateWord(frWord: str) -> str:
        if rng.random() < noiseProb:
            return str(rng.choice(enBase))
        return lexicon[frWord]

    def sampleLength() -> int:
        if rng.random() < 0.08:  # ~8% de queue longue, garantit des cibles > longLengthThreshold
            return int(rng.integers(longLengthThreshold + 1, longLengthThreshold + 12))
        return int(rng.integers(3, 11))  # cas courant : phrases courtes

    rows = []
    for _ in range(size):
        length = sampleLength()
        frWords = list(rng.choice(frVocabForSentences, size=length))
        enWords = [translateWord(w) for w in frWords]
        rows.append({srcLang: " ".join(frWords), tgtLang: " ".join(enWords)})

    corpus = pd.DataFrame(rows)

    # Duplication de cibles EN (alignement 1->N) : on prend une fraction des paires
    # dont la source contient un mot "base" ayant un synonyme, et on fabrique une
    # source FR alternative (mot remplacé par son synonyme) qui pointe vers LA MÊME
    # cible EN -- exactement le mécanisme "plusieurs sources FR, une cible EN" du
    # vrai corpus (cf. CLAUDE.md, 66 542 cibles EN dupliquées).
    dupFraction = 0.15
    hasSynonymBase = corpus[srcLang].apply(
        lambda s: any(f"fr{i:02d}" in s.split() for i in range(nSynonymPairs))
    )
    candidateIdx = corpus.index[hasSynonymBase].to_numpy()
    nDup = int(len(corpus) * dupFraction)
    dupIdx = rng.choice(candidateIdx, size=min(nDup, len(candidateIdx)), replace=False)

    dupRows = []
    for idx in dupIdx:
        frWords = corpus.loc[idx, srcLang].split()
        enTarget = corpus.loc[idx, tgtLang]  # cible EN inchangée : c'est la duplication voulue
        for i in range(nSynonymPairs):
            baseWord = f"fr{i:02d}"
            if baseWord in frWords:
                frWords[frWords.index(baseWord)] = f"frsyn{i:02d}"
                break
        dupRows.append({srcLang: " ".join(frWords), tgtLang: enTarget})

    corpus = pd.concat([corpus, pd.DataFrame(dupRows)], ignore_index=True)
    corpus = corpus.sample(frac=1.0, random_state=seed).reset_index(drop=True)  # mélange
    return corpus


def splitGroupedByTarget(
    df: pd.DataFrame, ratios: tuple[float, float, float], seed: int
) -> dict[str, pd.DataFrame]:
    """
    Split train/val/test groupé par cible EN (anti-fuite, cf. étape 3) : un groupe
    entier (toutes les paires qui partagent la même cible) va dans une seule part.
    Stub simplifié : pas de stratification par longueur ici -- elle est appliquée
    plus loin, sur le train, par `subsampleStratifiedByLength`.
    """
    rng = np.random.default_rng(seed)
    groupKeys = df[tgtLang].unique()
    shuffled = rng.permutation(groupKeys)

    nTrain = int(len(shuffled) * ratios[0])
    nVal = int(len(shuffled) * ratios[1])
    trainKeys = set(shuffled[:nTrain])
    valKeys = set(shuffled[nTrain : nTrain + nVal])
    testKeys = set(shuffled[nTrain + nVal :])

    return {
        "train": df[df[tgtLang].isin(trainKeys)].reset_index(drop=True),
        "val": df[df[tgtLang].isin(valKeys)].reset_index(drop=True),
        "test": df[df[tgtLang].isin(testKeys)].reset_index(drop=True),
    }


def loadSplitsStub(**kwargs) -> dict[str, pd.DataFrame]:
    """Stub des étapes 0→3 : corpus jouet + split groupé par cible EN (80/10/10).

    `**kwargs` (`data_dir`, `processed_dir`, ...) est accepté et IGNORÉ -- présent
    uniquement pour tolérer l'appel réel `loadSplits(data_dir=..., processed_dir=...)`.
    """
    del kwargs
    corpus = generateToyCorpus(toyCorpusSize, randomSeed)
    return splitGroupedByTarget(corpus, ratios=(0.8, 0.1, 0.1), seed=randomSeed)

In [ ]:
# Stub vocabulaire / numérisation / DataLoader (étapes 4 et 5)
SPECIAL_TOKENS = {"<pad>": 0, "<unk>": 1, "<sos>": 2, "<eos>": 3}


def buildVocabFromSeries(series: pd.Series) -> dict[str, int]:
    """Construit un vocabulaire {mot: indice}, tokens spéciaux fixés aux indices 0-3."""
    vocab = dict(SPECIAL_TOKENS)
    counts: dict[str, int] = {}
    for sentence in series:
        for word in sentence.split():
            counts[word] = counts.get(word, 0) + 1
    for word, _ in sorted(counts.items(), key=lambda kv: (-kv[1], kv[0])):
        vocab[word] = len(vocab)
    return vocab


def buildVocabsStub(
    trainDf: pd.DataFrame, config: str | None = None, **kwargs
) -> tuple[dict[str, int], dict[str, int]]:
    """Stub de l'étape 4 : vocabulaires FR/EN séparés, construits sur le TRAIN seul.

    `config` et `**kwargs` (`processed_dir`, `force_rebuild`, ...) sont acceptés et
    IGNORÉS (le stub ne connaît qu'un seul vocabulaire mots entiers) -- présents
    uniquement pour tolérer l'appel réel `buildVocabs(train, tokenizationConfig, processed_dir=...)`.
    """
    del config, kwargs
    frVocab = buildVocabFromSeries(trainDf[srcLang])
    enVocab = buildVocabFromSeries(trainDf[tgtLang])
    return frVocab, enVocab


def numerizeSentence(sentence: str, vocab: dict[str, int], targetLen: int) -> torch.Tensor:
    """Encode une phrase en indices `[<sos>] + mots + [<eos>]`, paddée/tronquée à targetLen."""
    unk = vocab["<unk>"]
    ids = [vocab["<sos>"]] + [vocab.get(w, unk) for w in sentence.split()] + [vocab["<eos>"]]
    ids = ids[:targetLen]
    ids = ids + [vocab["<pad>"]] * (targetLen - len(ids))
    return torch.tensor(ids, dtype=torch.long)


class ToyTranslationDataset(Dataset):
    """Dataset PyTorch pour des paires FR-EN déjà numérisées et paddées à maxLen."""

    def __init__(self, df: pd.DataFrame, frVocab: dict, enVocab: dict, targetLen: int) -> None:
        self.srcTensors = [numerizeSentence(s, frVocab, targetLen) for s in df[srcLang]]
        self.tgtTensors = [numerizeSentence(s, enVocab, targetLen) for s in df[tgtLang]]

    def __len__(self) -> int:
        return len(self.srcTensors)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        return self.srcTensors[idx], self.tgtTensors[idx]


def makeDataloadersStub(
    splits: dict[str, pd.DataFrame],
    frVocab: dict,
    enVocab: dict,
    targetLen: int | None,
    batchSizeArg: int,
) -> dict[str, DataLoader]:
    """Stub de l'étape 5 : Dataset + DataLoader numérisés et paddés à `targetLen`.

    `targetLen=None` -> retombe sur une valeur jouet fixe (16) : le stub ne fait pas
    la dérivation p99 de la vraie implémentation (`src.tokenization.numerize`), il ne
    fait que tolérer l'appel réel `makeDataloaders(searchSplits, frVocab, enVocab, maxLen, batchSize)`.
    """
    if targetLen is None:
        targetLen = 16  # valeur jouet, cf. docstring
    loaders = {}
    for name, df in splits.items():
        dataset = ToyTranslationDataset(df, frVocab, enVocab, targetLen)
        loaders[name] = DataLoader(dataset, batch_size=batchSizeArg, shuffle=(name == "train"))
    return loaders

In [ ]:
# Stub modèle (étape 6) : encodeur/décodeur, cellule RNN|GRU paramétrable, teacher forcing
class EncoderStub(nn.Module):
    """Embedding -> cellule récurrente paramétrable (RNN|GRU). Renvoie TOUS les états
    cachés (nécessaire pour greffer l'attention plus tard, cf. étape 9, sans refonte)."""

    def __init__(self, vocabSize: int, embDim: int, hiddenDim: int, cellType: str) -> None:
        super().__init__()
        self.embedding = nn.Embedding(vocabSize, embDim, padding_idx=SPECIAL_TOKENS["<pad>"])
        cellCls = {"rnn": nn.RNN, "gru": nn.GRU}[cellType]
        self.rnn = cellCls(embDim, hiddenDim, batch_first=True)

    def forward(self, src: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        embedded = self.embedding(src)
        outputs, hidden = self.rnn(embedded)  # outputs: (batch, seqLen, hiddenDim)
        return outputs, hidden


class DecoderStub(nn.Module):
    """Embedding -> cellule récurrente -> linéaire. Un pas de décodage à la fois,
    pour permettre le teacher forcing token par token dans `Seq2SeqStub`."""

    def __init__(
        self, vocabSize: int, embDim: int, hiddenDim: int, cellType: str, dropout: float
    ) -> None:
        super().__init__()
        self.embedding = nn.Embedding(vocabSize, embDim, padding_idx=SPECIAL_TOKENS["<pad>"])
        self.dropout = nn.Dropout(dropout)
        cellCls = {"rnn": nn.RNN, "gru": nn.GRU}[cellType]
        self.rnn = cellCls(embDim, hiddenDim, batch_first=True)
        self.out = nn.Linear(hiddenDim, vocabSize)

    def forwardStep(
        self, inputToken: torch.Tensor, hidden: torch.Tensor
    ) -> tuple[torch.Tensor, torch.Tensor]:
        embedded = self.dropout(self.embedding(inputToken))
        output, hidden = self.rnn(embedded, hidden)
        prediction = self.out(output.squeeze(1))
        return prediction, hidden


class Seq2SeqStub(nn.Module):
    """Ossature encodeur-décodeur (pas d'attention ici, cf. étape 9) avec
    `teacherForcingRatio` intégré : 1.0 = teacher forcing pur, 0.0 = génération libre."""

    def __init__(
        self,
        frVocabSize: int,
        enVocabSize: int,
        embDim: int,
        hiddenDim: int,
        cellType: str,
        dropout: float,
    ) -> None:
        super().__init__()
        self.encoder = EncoderStub(frVocabSize, embDim, hiddenDim, cellType)
        self.decoder = DecoderStub(enVocabSize, embDim, hiddenDim, cellType, dropout)
        self.enVocabSize = enVocabSize

    def forward(
        self, src: torch.Tensor, tgt: torch.Tensor, teacherForcingRatioArg: float
    ) -> torch.Tensor:
        batchSizeLocal, tgtLenSeq = tgt.shape
        tensorDevice = src.device
        _, hidden = self.encoder(src)

        outputs = torch.zeros(batchSizeLocal, tgtLenSeq, self.enVocabSize, device=tensorDevice)
        inputToken = tgt[:, 0:1]  # <sos>
        for t in range(1, tgtLenSeq):
            prediction, hidden = self.decoder.forwardStep(inputToken, hidden)
            outputs[:, t, :] = prediction
            useTeacherForcing = torch.rand(1).item() < teacherForcingRatioArg
            top1 = prediction.argmax(1, keepdim=True)
            inputToken = tgt[:, t : t + 1] if useTeacherForcing else top1
        return outputs


def buildModelStub(
    frVocabSize: int,
    enVocabSize: int,
    embDim: int,
    hiddenDim: int,
    cellType: str,
    dropout: float,
    use_attention: bool = False,
) -> Seq2SeqStub:
    """Stub de l'étape 6 : construit un `Seq2SeqStub` (RNN ou GRU selon `cellType`).

    `use_attention` est accepté et IGNORÉ (le stub ne modélise pas l'attention, cf. étape 9)
    -- présent uniquement pour tolérer l'appel réel `buildModel(..., use_attention=useAttention)`.
    """
    del use_attention
    return Seq2SeqStub(frVocabSize, enVocabSize, embDim, hiddenDim, cellType, dropout)

In [ ]:
# Stub boucle d'entraînement (étape 7) : loss ignorant le pad, clipping, callback de pruning
def trainAndValidateStub(
    model: nn.Module,
    loaders: dict[str, DataLoader],
    maxEpochsArg: int,
    lr: float,
    gradClipArg: float,
    teacherForcingRatioArg: float,
    patienceArg: int,
    deviceArg: torch.device,
    on_epoch_end: Callable[[int, float], bool] | None = None,
) -> dict:
    """
    Stub de l'étape 7 : `CrossEntropyLoss(ignore_index=<pad>)`, Adam, clipping de
    gradient, suivi train/val loss + perplexité. `on_epoch_end(epoch, valLoss)` est
    appelé à chaque fin d'epoch ; s'il renvoie `True`, l'entraînement s'arrête --
    c'est le point d'accroche du pruning Optuna (qui peut aussi interrompre en
    levant une exception depuis `onEpochEnd`, cf. `makeObjective`).
    """
    model.to(deviceArg)
    criterion = nn.CrossEntropyLoss(ignore_index=SPECIAL_TOKENS["<pad>"])
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history: dict[str, list] = {"trainLoss": [], "valLoss": [], "valPerplexity": []}
    bestValLoss = float("inf")
    epochsWithoutImprovement = 0

    for epoch in range(1, maxEpochsArg + 1):
        model.train()
        trainLossSum, nTrainBatches = 0.0, 0
        for src, tgt in loaders["train"]:
            src, tgt = src.to(deviceArg), tgt.to(deviceArg)
            optimizer.zero_grad()
            output = model(src, tgt, teacherForcingRatioArg)
            outputDim = output.shape[-1]
            loss = criterion(output[:, 1:, :].reshape(-1, outputDim), tgt[:, 1:].reshape(-1))
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), gradClipArg)
            optimizer.step()
            trainLossSum += loss.item()
            nTrainBatches += 1
        trainLoss = trainLossSum / max(nTrainBatches, 1)

        model.eval()
        valLossSum, nValBatches = 0.0, 0
        with torch.no_grad():
            for src, tgt in loaders["val"]:
                src, tgt = src.to(deviceArg), tgt.to(deviceArg)
                output = model(src, tgt, 0.0)  # pas de triche en validation
                outputDim = output.shape[-1]
                loss = criterion(output[:, 1:, :].reshape(-1, outputDim), tgt[:, 1:].reshape(-1))
                valLossSum += loss.item()
                nValBatches += 1
        valLoss = valLossSum / max(nValBatches, 1)
        valPerplexity = math.exp(min(valLoss, 20)) if math.isfinite(valLoss) else float("inf")

        history["trainLoss"].append(trainLoss)
        history["valLoss"].append(valLoss)
        history["valPerplexity"].append(valPerplexity)

        if valLoss < bestValLoss - 1e-4:
            bestValLoss = valLoss
            epochsWithoutImprovement = 0
        else:
            epochsWithoutImprovement += 1

        shouldStop = bool(on_epoch_end(epoch, valLoss)) if on_epoch_end is not None else False
        if shouldStop or epochsWithoutImprovement >= patienceArg:
            break

    return {"history": history, "bestValLoss": bestValLoss}

In [ ]:
# Sélection effective des fonctions (src/ si disponible, sinon stub) et chargement des données
loadSplits = pipelineInfo["resolved"].get("loadSplits", loadSplitsStub)
buildVocabs = pipelineInfo["resolved"].get("buildVocabs", buildVocabsStub)
makeDataloaders = pipelineInfo["resolved"].get("makeDataloaders", makeDataloadersStub)
buildModel = pipelineInfo["resolved"].get("buildModel", buildModelStub)
trainAndValidate = pipelineInfo["resolved"].get("trainAndValidate", trainAndValidateStub)

splits = loadSplits(data_dir="../data", processed_dir="../data/processed")
frVocab, enVocab = buildVocabs(splits["train"], tokenizationConfig, processed_dir="../data/processed")

print(
    f"Split -- train: {len(splits['train'])} paires, val: {len(splits['val'])} paires, "
    f"test: {len(splits['test'])} paires"
)
print(f"Vocabulaire FR: {len(frVocab)} tokens | Vocabulaire EN: {len(enVocab)} tokens")

## 2. Sous-échantillon 20 % stratifié par longueur

Étape 8 du plan : la recherche Optuna tourne sur un sous-ensemble d'environ 20 % du
train, pour aller vite, **sans perdre les phrases longues** (c'est sur elles que
l'écart RNN/GRU se joue). Le regroupement se fait par **cible EN** (jamais couper un
groupe), cohérent avec l'anti-fuite de l'étape 3 : un groupe dupliqué (alignement
1→N) reste entier dans le sous-ensemble ou en dehors, jamais réparti des deux côtés.

In [ ]:
# Sous-échantillonnage stratifié par longueur, groupé par cible EN
def subsampleStratifiedByLength(
    df: pd.DataFrame, fraction: float, threshold: int, seed: int
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Sous-échantillonne `df` en conservant la proportion de phrases longues. Regroupe
    par cible EN (jamais couper un groupe en deux, cohérent avec l'anti-fuite de
    l'étape 3), tire aléatoirement des GROUPES entiers dans chaque strate (longueur
    de groupe = nb de mots de sa cible EN, seuil `threshold`), puis reconstitue les
    paires. Renvoie (sous-ensemble, diagnostic avant/après).
    """
    rng = np.random.default_rng(seed)
    uniqueTargets = df[tgtLang].unique()
    lengthByTarget = {t: len(t.split()) for t in uniqueTargets}

    shortKeys = np.array([t for t in uniqueTargets if lengthByTarget[t] <= threshold], dtype=object)
    longKeys = np.array([t for t in uniqueTargets if lengthByTarget[t] > threshold], dtype=object)

    def sampleGroupKeys(keys: np.ndarray) -> np.ndarray:
        if len(keys) == 0:
            return keys
        nKeep = min(max(1, int(round(len(keys) * fraction))), len(keys))
        return rng.choice(keys, size=nKeep, replace=False)

    keptKeys = set(sampleGroupKeys(shortKeys)) | set(sampleGroupKeys(longKeys))
    subset = df[df[tgtLang].isin(keptKeys)].reset_index(drop=True)

    def diagnosticRow(label: str, frame: pd.DataFrame) -> dict:
        keys = frame[tgtLang].unique()
        nLongues = sum(1 for k in keys if lengthByTarget[k] > threshold)
        nGroupes = len(keys)
        return {
            "jeu": label,
            "nPaires": len(frame),
            "nGroupes": nGroupes,
            "nLongues": nLongues,
            "proportionLongues": nLongues / nGroupes if nGroupes else 0.0,
        }

    diagnostic = pd.DataFrame([diagnosticRow("avant", df), diagnosticRow("après", subset)])
    return subset, diagnostic

In [ ]:
# Application du sous-échantillonnage au train + points de contrôle + DataLoaders
trainSubsample, subsampleDiag = subsampleStratifiedByLength(
    splits["train"], subsampleFraction, longLengthThreshold, randomSeed
)
display(subsampleDiag)

nLonguesApres = int(subsampleDiag.loc[subsampleDiag["jeu"] == "après", "nLongues"].iloc[0])
assert nLonguesApres > 0, (
    "Le sous-ensemble stratifié ne contient aucune phrase longue -- "
    "vérifier longLengthThreshold ou le corpus jouet."
)

propAvant = float(subsampleDiag.loc[subsampleDiag["jeu"] == "avant", "proportionLongues"].iloc[0])
propApres = float(subsampleDiag.loc[subsampleDiag["jeu"] == "après", "proportionLongues"].iloc[0])
print(
    f"Proportion de groupes longs -- avant: {propAvant:.1%}, après: {propApres:.1%} "
    f"(écart {abs(propApres - propAvant):.1%})"
)
print(f"Sous-ensemble retenu pour Optuna: {len(trainSubsample)} paires ({nLonguesApres} groupes longs).")

# DataLoaders pour la recherche : train sous-échantillonné, val/test complets
# (val/test ne sont pas sous-échantillonnés : ils servent à mesurer, pas à accélérer)
searchSplits = {"train": trainSubsample, "val": splits["val"], "test": splits["test"]}
dataloaders = makeDataloaders(searchSplits, frVocab, enVocab, maxLen, batchSize)
print(
    f"DataLoaders prêts -- train: {len(dataloaders['train'].dataset)}, "
    f"val: {len(dataloaders['val'].dataset)}, test: {len(dataloaders['test'].dataset)}"
)

## 3. Espace de recherche

Restreint aux hyperparamètres qui **ne changent pas les données** : `learningRate`
(le plus déterminant, ~80 % de l'effet selon le plan), `hiddenDim`, `embDim`,
`dropout`. `cellType` (RNN/GRU) est **hors** de cet espace : il fait l'objet de deux
études séparées (option A du plan, comparaison « meilleur plafond vs meilleur
plafond »). Le tokeniseur/vocabulaire est également hors espace : il change les
données elles-mêmes, il est exploré séparément à l'étape 8bis (hors Optuna).

In [ ]:
# Espace de recherche des hyperparamètres
def suggestHyperparams(trial: optuna.Trial) -> dict:
    """Échantillonne un jeu d'hyperparamètres pour un essai Optuna."""
    return {
        "learningRate": trial.suggest_float("learningRate", *lrRange, log=True),
        "hiddenDim": trial.suggest_categorical("hiddenDim", hiddenDimChoices),
        "embDim": trial.suggest_categorical("embDim", embDimChoices),
        "dropout": trial.suggest_float("dropout", *dropoutRange, step=dropoutStep),
    }

## 4. Fonction objectif et pruning

`makeObjective(cellType, ...)` construit une closure `objective(trial)` qui entraîne
un modèle avec les hyperparamètres suggérés, rapporte la val loss à Optuna à chaque
epoch (`trial.report`) et lève `optuna.TrialPruned` si `trial.should_prune()` ou si
la val loss devient NaN/inf (essai raté, à couper immédiatement plutôt que de
gaspiller du budget).

In [ ]:
# Fonction objectif Optuna (fermeture sur l'architecture et les données)
def makeObjective(
    cellType: str,
    loadersArg: dict[str, DataLoader],
    frVocabSize: int,
    enVocabSize: int,
    deviceArg: torch.device,
) -> Callable[[optuna.Trial], float]:
    """
    Construit la fonction objectif Optuna pour l'architecture `cellType`. Renvoie la
    meilleure val loss atteinte (direction "minimize").
    """

    def objective(trial: optuna.Trial) -> float:
        hp = suggestHyperparams(trial)
        torch.manual_seed(randomSeed + trial.number)  # reproductible mais différent par essai

        model = buildModel(
            frVocabSize,
            enVocabSize,
            hp["embDim"],
            hp["hiddenDim"],
            cellType,
            hp["dropout"],
            use_attention=useAttention,
        )

        def onEpochEnd(epoch: int, valLoss: float) -> bool:
            if not math.isfinite(valLoss):
                raise optuna.TrialPruned(f"val loss non finie à l'epoch {epoch}")
            trial.report(valLoss, epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()
            return False

        result = trainAndValidate(
            model,
            loadersArg,
            maxEpochs,
            hp["learningRate"],
            gradClip,
            teacherForcingRatio,
            patience,
            deviceArg,
            on_epoch_end=onEpochEnd,
        )

        bestValLoss = result["bestValLoss"]
        if not math.isfinite(bestValLoss):
            raise optuna.TrialPruned("val loss finale non finie")
        return bestValLoss

    return objective

## 5. Lancement des études (option A : une étude par architecture)

`runStudy(cellType)` crée (ou reprend, `load_if_exists=True`) une étude Optuna par
architecture, stockée en SQLite (`checkpoints/optuna_rosetta.db`) pour pouvoir
interrompre puis reprendre la recherche sans perdre les essais déjà faits.

In [ ]:
# Lancement (ou reprise) d'une étude Optuna pour une architecture donnée
def optunaTrialCallback(study: optuna.Study, trial: optuna.trial.FrozenTrial) -> None:
    """Callback de logging compact : une ligne par essai terminé."""
    value = trial.value if trial.value is not None else float("nan")
    print(f"  [essai {trial.number:>3}] état={trial.state.name:<9} valeur={value:.4f} params={trial.params}")


def runStudy(
    cellType: str,
    loadersArg: dict[str, DataLoader],
    frVocabSize: int,
    enVocabSize: int,
    deviceArg: torch.device,
) -> optuna.Study:
    """
    Lance (ou reprend) l'étude Optuna pour l'architecture `cellType`.

    Le nom d'étude inclut `studyNameSuffix` ("-stub" en mode jouet, "" en mode réel) :
    sans ce suffixe, `load_if_exists=True` ferait reprendre une exécution réelle par
    l'étude qui contient déjà les essais du corpus jouet, et `trials_dataframe()` /
    `best_params` mélangeraient silencieusement deux corpus différents.
    """
    Path(storageDir).mkdir(parents=True, exist_ok=True)
    storageUrl = f"sqlite:///{storageDir}/{studyStorageName}"
    sampler = optuna.samplers.TPESampler(seed=randomSeed)
    pruner = optuna.pruners.MedianPruner(n_startup_trials=nStartupTrials, n_warmup_steps=nWarmupSteps)

    study = optuna.create_study(
        study_name=f"rosetta-{cellType}{studyNameSuffix}",
        storage=storageUrl,
        load_if_exists=True,
        direction="minimize",
        sampler=sampler,
        pruner=pruner,
    )

    objective = makeObjective(cellType, loadersArg, frVocabSize, enVocabSize, deviceArg)
    print(
        f"Étude '{study.study_name}' -- {len(study.trials)} essai(s) déjà enregistré(s), "
        f"lancement de {nTrials} essai(s) additionnel(s)."
    )
    study.optimize(objective, n_trials=nTrials, callbacks=[optunaTrialCallback])
    return study

In [ ]:
# Boucle sur les architectures (option A : une étude complète par architecture)
optuna.logging.set_verbosity(optuna.logging.WARNING)

studies: dict[str, optuna.Study] = {}
for cellType in architectures:
    print(f"\n=== Étude Optuna -- architecture: {cellType} ===")
    studies[cellType] = runStudy(cellType, dataloaders, len(frVocab), len(enVocab), device)

## 6. Résultats et points de contrôle

Export des essais (`trials_dataframe()`), des meilleurs hyperparamètres par
architecture, d'un tableau comparatif RNN vs GRU, et des figures de diagnostic
Optuna (historique d'optimisation, importance des hyperparamètres, coordonnées
parallèles) -- via le backend matplotlib d'Optuna (`plotly` n'est pas installé).

In [ ]:
# Export des essais loggés (trials_dataframe) en CSV, par architecture
Path(runReportsDir).mkdir(parents=True, exist_ok=True)

trialsDataframes: dict[str, pd.DataFrame] = {}
for cellType, study in studies.items():
    trialsDf = study.trials_dataframe()
    trialsDataframes[cellType] = trialsDf
    outPath = Path(runReportsDir) / f"trials_{cellType}.csv"
    trialsDf.to_csv(outPath, index=False)
    print(f"{cellType}: {len(trialsDf)} essai(s) loggé(s) -> {outPath}")

In [ ]:
# Meilleurs hyperparamètres par architecture (JSON) + tableau comparatif (CSV)
comparisonRows = []
for cellType, study in studies.items():
    completedTrials = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    prunedTrials = [t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]

    bestParamsRecord = {
        "cellType": cellType,
        "bestValLoss": study.best_value,
        "bestParams": study.best_params,
        "nTrialsCompleted": len(completedTrials),
        "nTrialsPruned": len(prunedTrials),
        "nTrialsTotal": len(study.trials),
        "pipelineReady": pipelineReady,
    }
    outPath = Path(runReportsDir) / f"best_params_{cellType}.json"
    with open(outPath, "w", encoding="utf-8") as f:
        json.dump(bestParamsRecord, f, ensure_ascii=False, indent=2)
    print(f"{cellType}: meilleure val loss = {study.best_value:.4f} -> {outPath}")

    comparisonRows.append(
        {
            "cellType": cellType,
            "bestValLoss": study.best_value,
            **study.best_params,
            "nCompleted": len(completedTrials),
            "nPruned": len(prunedTrials),
        }
    )

comparisonDf = pd.DataFrame(comparisonRows)
comparisonPath = Path(runReportsDir) / "comparaison_studies.csv"
comparisonDf.to_csv(comparisonPath, index=False)
display(comparisonDf)
print(f"Tableau comparatif RNN vs GRU -> {comparisonPath}")

In [ ]:
# Figures de diagnostic Optuna (backend matplotlib, PedAnova pour l'importance)
for cellType, study in studies.items():
    try:
        ax = optunaViz.plot_optimization_history(study)
        ax.figure.savefig(Path(runReportsDir) / f"fig_history_{cellType}.png", bbox_inches="tight")
        plt.close(ax.figure)
    except Exception as exc:  # noqa: BLE001 -- on veut continuer même si une figure échoue
        print(f"[avertissement] figure d'historique indisponible pour {cellType}: {exc}")

    try:
        evaluator = PedAnovaImportanceEvaluator()  # pur numpy, pas besoin de scikit-learn
        ax = optunaViz.plot_param_importances(study, evaluator=evaluator)
        ax.figure.savefig(Path(runReportsDir) / f"fig_importance_{cellType}.png", bbox_inches="tight")
        plt.close(ax.figure)
    except Exception as exc:  # noqa: BLE001
        print(f"[avertissement] figure d'importance indisponible pour {cellType}: {exc}")

    try:
        ax = optunaViz.plot_parallel_coordinate(study)
        ax.figure.savefig(Path(runReportsDir) / f"fig_parallel_{cellType}.png", bbox_inches="tight")
        plt.close(ax.figure)
    except Exception as exc:  # noqa: BLE001
        print(f"[avertissement] figure des coordonnées parallèles indisponible pour {cellType}: {exc}")

In [ ]:
# Checklist -- points de contrôle de l'étape 8 du plan
print("=" * 74)
print("Checklist -- points de contrôle de l'étape 8 du plan")
print("=" * 74)

checks = []

# 1. Le sous-ensemble 20 % contient bien des phrases longues.
checks.append(("Le sous-ensemble 20 % contient des phrases longues", nLonguesApres > 0))

# 2. Tous les essais sont loggés (trials_dataframe non vide, une ligne par essai).
allTrialsLogged = all(
    len(trialsDataframes[c]) == len(studies[c].trials) and len(trialsDataframes[c]) > 0
    for c in architectures
)
checks.append(("Tous les essais sont loggés (trials_dataframe)", allTrialsLogged))

# 3. Meilleurs hyperparamètres récupérés par architecture (fichiers JSON présents).
bestParamsExported = all(
    (Path(runReportsDir) / f"best_params_{c}.json").exists() for c in architectures
)
checks.append(("Meilleurs hyperparamètres exportés par architecture", bestParamsExported))

for label, ok in checks:
    print(f"[{'OK' if ok else 'KO'}] {label}")

assert all(ok for _, ok in checks), "Au moins un point de contrôle de l'étape 8 a échoué."
print("\nTous les points de contrôle de l'étape 8 sont validés.")

## 7. Limites et suite

- **20 % ≠ optimum du dataset complet** : les hyperparamètres réglés sur le
  sous-ensemble stratifié sont un bon point de départ, pas l'optimum exact du
  dataset complet -- léger risque de sur-régularisation (dropout, notamment) réglé
  pour un jeu plus petit que celui de l'entraînement final (étape 9).
- **Tokeniseur volontairement hors de l'espace de recherche** : il change les
  données (pas seulement le modèle), donc il est exploré séparément et sans
  prétention de comparaison contrôlée à l'étape 8bis (mots entiers 95 % vs
  sous-mots Unigram).
- **Brancher `src/` ne demande aucune modification de ce notebook** : implémenter
  les 5 symboles attendus (`loadSplits`, `buildVocabs`, `makeDataloaders`,
  `buildModel`, `trainAndValidate`) dans `src/data`, `src/tokenization`,
  `src/models`, `src/training` suffit -- `resolvePipeline()` les détecte et les
  stubs se désactivent seuls (`pipelineReady` passe à `True`).
- **Budget de ce run** : calibré pour un smoke-test rapide sur corpus jouet
  (`nTrials`, `maxEpochs`, `toyCorpusSize` réduits, cf. cellule Paramètres). À
  remonter (`nTrials` 30-50, `maxEpochs` 15-20) une fois `src/` branché et le run
  exécuté sur le vrai sous-ensemble 20 %.
- **Séparation de provenance jouet/réel** : les artefacts jamais mélangés --
  dossier `reports/optuna/stub/` et suffixe d'étude `-stub` tant que
  `pipelineReady` est faux, bascule automatique vers `reports/optuna/` et les
  études `rosetta-rnn`/`rosetta-gru` dès que les 5 symboles sont résolus depuis
  `src/`. Le paramètre `forceStubs` permet de forcer le mode jouet même si
  `src/` est déjà (partiellement) branché.